# Creer l'APK EGM OS (tout-en-un)
Ce fichier contient deja tout le code. Lancez simplement chaque case dans l'ordre
(bouton lecture a gauche). La case 1 est tres longue (30 min a 1 h), c'est normal.
A la fin, l'APK se telecharge tout seul sur votre ordinateur.

In [ ]:
# Case 1 : Installation des outils (quelques minutes)
!pip install -q buildozer cython==0.29.33
!mkdir -p /content/egm-os
%cd /content/egm-os

In [ ]:
# Case 2 : Ecriture du code d'EGM OS (automatique)

In [ ]:
%%writefile main.py
"""EGM OS - Lanceur principalUn mini bureau en Python avec 4 options :  1. EGM Download (gestionnaire de telechargements integre)  2. Lecteur multimedia (mp4, mp3, m4v, wav, vob, mov, mkv, avi, ogg)  3. Navigateur web  4. Telechargeur de videos (style downloadhelper.net)Multiplateforme : Windows, Linux (et Android via Pydroid 3).Lancez avec :  python main.py"""import osimport sysimport subprocessimport tkinter as tkfrom tkinter import messageboxBG = "#0f1420"FG = "#e8ecf4"ACCENT = "#1e88e5"CARD = "#1a2233"HOVER = "#243049"TITLES = [    ("1", "EGM Download", "Gestionnaire de telechargements integre", "download"),    ("2", "Lecteur Multimedia", "mp4, mp3, m4v, wav, vob, mov, mkv, avi, ogg", "play"),    ("3", "Navigateur Web", "Surfer sur Internet", "globe"),    ("4", "Telechargeur Video", "Capturer des videos depuis le web", "capture"),]def open_module(choice: str) -> None:    try:        if choice == "download":            import downloader            downloader.run()        elif choice == "play":            import player            player.run()        elif choice == "globe":            import browser            browser.run()        elif choice == "capture":            import downloader            downloader.run(capture_mode=True)    except Exception as exc:  # Surface les erreurs dans l'UI        messagebox.showerror("EGM OS", f"Impossible d'ouvrir ce module :\n{exc}")def make_card(root: tk.Widget, num: str, title: str, subtitle: str, key: str) -> tk.Frame:    card = tk.Frame(root, bg=CARD, highlightbackground=ACCENT, highlightthickness=0)    card.pack(fill="x", padx=30, pady=9, ipady=6)    num_lbl = tk.Label(card, text=num, bg=CARD, fg=ACCENT,                       font=("Segoe UI", 20, "bold"), width=3)    num_lbl.pack(side="left", padx=(18, 6))    txt = tk.Frame(card, bg=CARD)    txt.pack(side="left", fill="x", expand=True)    tk.Label(txt, text=title, bg=CARD, fg=FG,             font=("Segoe UI", 14, "bold"), anchor="w").pack(fill="x")    tk.Label(txt, text=subtitle, bg=CARD, fg="#9aa5b8",             font=("Segoe UI", 10), anchor="w").pack(fill="x")    def on_enter(_event):        card.configure(bg=HOVER, highlightthickness=1)        for w in card.winfo_children():            if isinstance(w, tk.Frame):                for c in w.winfo_children():                    c.configure(bg=HOVER)            else:                w.configure(bg=HOVER)    def on_leave(_event):        card.configure(bg=CARD, highlightthickness=0)        for w in card.winfo_children():            if isinstance(w, tk.Frame):                for c in w.winfo_children():                    c.configure(bg=CARD)            else:                w.configure(bg=CARD)    def on_click(_event):        open_module(key)    for widget in (card, num_lbl, txt):        widget.bind("<Enter>", on_enter)        widget.bind("<Leave>", on_leave)        widget.bind("<Button-1>", on_click)    for w in txt.winfo_children():        w.bind("<Enter>", on_enter)        w.bind("<Leave>", on_leave)        w.bind("<Button-1>", on_click)    return carddef run() -> None:    root = tk.Tk()    root.title("EGM OS")    root.geometry("640x560")    root.minsize(520, 480)    root.configure(bg=BG)    header = tk.Frame(root, bg=BG)    header.pack(fill="x", pady=(28, 6))    tk.Label(header, text="EGM OS", bg=BG, fg=FG,             font=("Segoe UI", 26, "bold")).pack()    tk.Label(header, text="Choisissez une option pour demarrer", bg=BG,             fg="#9aa5b8", font=("Segoe UI", 11)).pack(pady=(4, 14))    for num, title, subtitle, key in TITLES:        make_card(root, num, title, subtitle, key)    footer = tk.Frame(root, bg=BG)    footer.pack(side="bottom", fill="x", pady=10)    tk.Label(footer, text="EGM OS  -  Windows / Linux / Android (Pydroid 3)",             bg=BG, fg="#5d6a80", font=("Segoe UI", 9)).pack()    root.mainloop()if __name__ == "__main__":    run()

In [ ]:
%%writefile browser.py
"""EGM OS - Navigateur WebUtilise pywebview s'il est installe (vraie fenetre de navigation avec barred'adresse), sinon ouvre l'adresse dans le navigateur par defaut du systeme."""import sysimport threadingimport tkinter as tkfrom tkinter import messageboximport webbrowserBG = "#0f1420"FG = "#e8ecf4"ACCENT = "#1e88e5"CARD = "#1a2233"try:    import webview  # type: ignore    HAS_WEBVIEW = Trueexcept Exception:    HAS_WEBVIEW = FalseHOME_URL = "https://www.google.com"def normalize_url(text: str) -> str:    text = text.strip()    if not text:        return HOME_URL    if text.startswith("http://") or text.startswith("https://"):        return text    if " " not in text and "." in text:        return "https://" + text    return "https://duckduckgo.com/?q=" + text.replace(" ", "+")def launch_webview(url: str) -> None:    window = webview.create_window("EGM OS - Navigateur", url, width=1100, height=700)    webview.start(window)class TkBrowser:    """Navigateur de secours en pur Tkinter : barre d'adresse + navigation    dans le navigateur systeme (aucune dependance requise)."""    def __init__(self, root: tk.Tk) -> None:        self.root = root        self.history: list[str] = []        root.title("EGM OS - Navigateur Web")        root.geometry("640x300")        root.minsize(520, 240)        root.configure(bg=BG)        tk.Label(root, text="Adresse ou recherche", bg=BG, fg=FG,                 font=("Segoe UI", 12, "bold")).pack(pady=(18, 6))        bar = tk.Frame(root, bg=BG)        bar.pack(fill="x", padx=24)        self.entry = tk.Entry(bar, bg=CARD, fg=FG, insertbackground=FG,                              relief="flat", font=("Segoe UI", 12))        self.entry.pack(side="left", fill="x", expand=True, ipady=8, padx=(0, 8))        self.entry.insert(0, HOME_URL)        self.entry.bind("<Return>", lambda _e: self.go())        tk.Button(bar, text="Aller", command=self.go, bg=ACCENT, fg="white",                  relief="flat", font=("Segoe UI", 11, "bold"),                  padx=16).pack(side="right")        self.status = tk.Label(root, text="", bg=BG, fg="#9aa5b8",                               font=("Segoe UI", 9))        self.status.pack(side="bottom", pady=10)        note = ("pywebview non installe : les pages s'ouvrent dans votre "                "navigateur systeme.\nInstallez pywebview pour un navigateur "                "integre complet.")        tk.Label(root, text=note, bg=BG, fg="#5d6a80",                 font=("Segoe UI", 9), justify="center").pack(pady=16)    def go(self) -> None:        url = normalize_url(self.entry.get())        self.history.append(url)        webbrowser.open_new_tab(url)        self.status.configure(text="Ouvert : " + url)def run() -> None:    if HAS_WEBVIEW:        launch_webview(HOME_URL)        return    root = tk.Tk()    TkBrowser(root)    root.mainloop()

In [ ]:
%%writefile downloader.py
"""EGM OS - EGM Download / Telechargeur VideoTelecharge des videos depuis n'importe quel site (YouTube, Twitter, etc.)en style "downloadhelper.net" :  - collez l'URL de la page video  - choisissez le format (MP4 video ou MP3 audio)  - choisissez le dossier de destination  - la progression s'affiche en directNecessite : pip install yt-dlp"""import osimport shutilimport subprocessimport sysimport threadingimport tkinter as tkfrom tkinter import filedialog, messageboxBG = "#0f1420"FG = "#e8ecf4"ACCENT = "#1e88e5"CARD = "#1a2233"GREEN = "#2e9e5b"RED = "#d64545"def yt_dlp_available() -> bool:    return shutil.which("yt-dlp") is not None or _importable("yt_dlp")def _importable(module: str) -> bool:    try:        __import__(module)        return True    except Exception:        return Falseclass DownloaderWindow:    def __init__(self, root: tk.Tk, capture_mode: bool = False) -> None:        self.root = root        self.folder = os.path.join(os.path.expanduser("~"), "Downloads")        self.downloading = False        title = ("EGM Download" if not capture_mode                 else "Telechargeur Video (style downloadhelper.net)")        root.title("EGM OS - " + title)        root.geometry("680x480")        root.minsize(560, 420)        root.configure(bg=BG)        tk.Label(root, text=title, bg=BG, fg=FG,                 font=("Segoe UI", 18, "bold")).pack(pady=(20, 4))        tk.Label(root, text="Collez l'adresse de la page qui contient la video",                 bg=BG, fg="#9aa5b8", font=("Segoe UI", 10)).pack()        url_frame = tk.Frame(root, bg=BG)        url_frame.pack(fill="x", padx=24, pady=(16, 6))        tk.Label(url_frame, text="URL de la video", bg=BG, fg=FG,                 font=("Segoe UI", 10, "bold")).pack(anchor="w")        self.url_entry = tk.Entry(url_frame, bg=CARD, fg=FG, relief="flat",                                  font=("Segoe UI", 11), insertbackground=FG)        self.url_entry.pack(fill="x", ipady=9, pady=(4, 0))        fmt_frame = tk.Frame(root, bg=BG)        fmt_frame.pack(fill="x", padx=24, pady=6)        self.format_var = tk.StringVar(value="mp4")        tk.Label(fmt_frame, text="Format", bg=BG, fg=FG,                 font=("Segoe UI", 10, "bold")).pack(anchor="w")        row = tk.Frame(fmt_frame, bg=BG)        row.pack(anchor="w", pady=(4, 0))        for value, label in (("mp4", "MP4 (video)"), ("mp3", "MP3 (audio)"),                             ("best", "Meilleure qualite")):            tk.Radiobutton(row, text=label, variable=self.format_var,                           value=value, bg=BG, fg=FG, selectcolor=CARD,                           activebackground=BG, activeforeground=FG,                           font=("Segoe UI", 10)).pack(side="left", padx=(0, 16))        dir_frame = tk.Frame(root, bg=BG)        dir_frame.pack(fill="x", padx=24, pady=6)        tk.Label(dir_frame, text="Dossier de destination", bg=BG, fg=FG,                 font=("Segoe UI", 10, "bold")).pack(anchor="w")        dir_row = tk.Frame(dir_frame, bg=BG)        dir_row.pack(fill="x", pady=(4, 0))        self.dir_label = tk.Label(dir_row, text=self.folder, bg=CARD, fg=FG,                                  anchor="w", relief="flat")        self.dir_label.pack(side="left", fill="x", expand=True, ipady=7)        tk.Button(dir_row, text="Parcourir", command=self.pick_folder,                  bg=CARD, fg=FG, relief="flat", padx=12).pack(side="right",                                                               padx=(8, 0))        tk.Button(root, text="Telecharger", command=self.start_download,                  bg=ACCENT, fg="white", relief="flat",                  font=("Segoe UI", 13, "bold"), pady=8).pack(pady=14,                                                              ipadx=30)        self.progress = tk.DoubleVar(value=0.0)        bar = tk.Frame(root, bg=BG)        bar.pack(fill="x", padx=24)        tk.Label(bar, text="Progression", bg=BG, fg=FG,                 font=("Segoe UI", 9)).pack(anchor="w")        tk.Scale(bar, variable=self.progress, from_=0, to=100, orient="horizontal",                 bg=BG, fg=FG, highlightthickness=0, state="disabled",                 troughcolor=CARD, sliderrelief="flat").pack(fill="x")        self.status_label = tk.Label(root, text="", bg=BG, fg="#9aa5b8",                                     font=("Segoe UI", 10), wraplength=600)        self.status_label.pack(pady=(6, 12), padx=24)        if not yt_dlp_available():            self.status_label.configure(                text="Outil de telechargement absent. Installez-le avec : "                     "pip install yt-dlp",                fg="#e0a63a")    def pick_folder(self) -> None:        folder = filedialog.askdirectory(title="Dossier de destination")        if folder:            self.folder = folder            self.dir_label.configure(text=folder)    def start_download(self) -> None:        if self.downloading:            return        url = self.url_entry.get().strip()        if not url:            messagebox.showwarning("EGM Download", "Collez d'abord une URL.")            return        if not yt_dlp_available():            messagebox.showerror(                "EGM Download",                "L'outil yt-dlp n'est pas installe.\n"                "Ouvrez un terminal et lancez :  pip install yt-dlp")            return        self.downloading = True        self.progress.set(0)        self.status_label.configure(text="Telechargement en cours...", fg="#9aa5b8")        fmt = self.format_var.get()        args = ["yt-dlp", "--newline", "--no-playlist", "-o",                os.path.join(self.folder, "%(title)s.%(ext)s")]        if fmt == "mp3":            args += ["-x", "--audio-format", "mp3"]        elif fmt == "mp4":            args += ["-f", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best"]        args.append(url)        self._proc = subprocess.Popen(            args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,            text=True, encoding="utf-8", errors="replace")        threading.Thread(target=self._read_output, daemon=True).start()    def _read_output(self) -> None:        assert self._proc.stdout is not None        for line in self._proc.stdout:            line = line.strip()            if "[download]" in line and "%" in line:                try:                    pct = float(line.split("%")[0].split()[-1])                    self.root.after(0, self.progress.set, pct)                except ValueError:                    pass            elif line.startswith("ERROR"):                self.root.after(0, lambda l=line: self.status_label.configure(                    text=l, fg=RED))        code = self._proc.wait()        self.downloading = False        def done() -> None:            if code == 0:                self.progress.set(100)                self.status_label.configure(                    text="Termine ! Fichier enregistre dans " + self.folder,                    fg=GREEN)            else:                self.status_label.configure(                    text="Echec du telechargement. Verifiez l'URL et votre "                         "connexion.", fg=RED)        self.root.after(0, done)def run(capture_mode: bool = False) -> None:    root = tk.Tk()    DownloaderWindow(root, capture_mode=capture_mode)    root.mainloop()

In [ ]:
%%writefile player.py
"""EGM OS - Lecteur MultimediaFormats : mp4, mp3, m4v, wav, vob, mov, mkv, avi, oggUtilise python-vlc s'il est installe (lecture encastree), sinon ouvre lefichier avec le lecteur par defaut du systeme."""import osimport sysimport subprocessimport tkinter as tkfrom tkinter import filedialog, messageboxBG = "#0f1420"FG = "#e8ecf4"ACCENT = "#1e88e5"CARD = "#1a2233"FORMATS = [    ("Fichiers multimedia", "*.mp4 *.mp3 *.m4v *.wav *.vob *.mov *.mkv *.avi *.ogg *.mpg *.mpeg *.webm *.flac *.aac"),    ("Tous les fichiers", "*.*"),]try:    import vlc  # type: ignore    HAS_VLC = Trueexcept Exception:    HAS_VLC = Falsedef open_with_system(path: str) -> None:    """Ouvre le fichier avec le lecteur par defaut."""    if sys.platform.startswith("win"):        os.startfile(path)  # type: ignore[attr-defined]    elif sys.platform == "darwin":        subprocess.Popen(["open", path])    else:        subprocess.Popen(["xdg-open", path])class PlayerWindow:    def __init__(self, root: tk.Tk) -> None:        self.root = root        self.vlc_player = None        self.vlc_media = None        root.title("EGM OS - Lecteur Multimedia")        root.geometry("820x560")        root.minsize(600, 420)        root.configure(bg=BG)        top = tk.Frame(root, bg=BG)        top.pack(fill="x", pady=(14, 8), padx=16)        tk.Button(top, text="Ouvrir un fichier", command=self.pick_file,                  bg=ACCENT, fg="white", relief="flat",                  font=("Segoe UI", 11, "bold"), padx=14, pady=6).pack(side="left")        status = ("Lecture encaastree (python-vlc)" if HAS_VLC                  else "python-vlc non installe : ouverture avec le lecteur du systeme")        tk.Label(top, text=status, bg=BG,                 fg="#9aa5b8", font=("Segoe UI", 9)).pack(side="right")        self.file_label = tk.Label(root, text="Aucun fichier charge", bg=CARD,                                   fg=FG, font=("Segoe UI", 10), anchor="w")        self.file_label.pack(fill="x", padx=16, ipady=8)        self.video_area = tk.Frame(root, bg="#000000", height=320)        self.video_area.pack(fill="both", expand=True, padx=16, pady=10)        self.video_area.pack_propagate(False)        controls = tk.Frame(root, bg=BG)        controls.pack(fill="x", padx=16, pady=(0, 14))        for label, cmd in (("Play", self.play), ("Pause", self.pause),                           ("Stop", self.stop)):            tk.Button(controls, text=label, command=cmd, bg=CARD, fg=FG,                      relief="flat", font=("Segoe UI", 11, "bold"),                      padx=18, pady=6).pack(side="left", padx=(0, 8))        if HAS_VLC:            self.vlc_player = vlc.Instance()        self.current_path = None    def pick_file(self) -> None:        path = filedialog.askopenfilename(title="Choisir un fichier multimedia",                                          filetypes=FORMATS)        if not path:            return        self.current_path = path        self.file_label.configure(text=os.path.basename(path))        if HAS_VLC and self.vlc_player is not None:            self.vlc_media = self.vlc_player.media_new(path)            self.play()        else:            open_with_system(path)    def play(self) -> None:        if not HAS_VLC or self.vlc_player is None:            if self.current_path:                open_with_system(self.current_path)            return        if self.vlc_media is None:            messagebox.showinfo("Lecteur", "Ouvrez d'abord un fichier.")            return        self.vlc_media.parse()        player = self.vlc_player.media_player_new()        player.set_media(self.vlc_media)        self._player = player        self.root.after(300, lambda: self._attach_and_play(player))    def _attach_and_play(self, player) -> None:        if sys.platform.startswith("win"):            player.set_hwnd(self.video_area.winfo_id())        else:            player.set_xwindow(self.video_area.winfo_id())        player.play()    def pause(self) -> None:        if getattr(self, "_player", None):            self._player.pause()    def stop(self) -> None:        if getattr(self, "_player", None):            self._player.stop()    def close(self) -> None:        self.stop()def run() -> None:    root = tk.Tk()    win = PlayerWindow(root)    root.protocol("WM_DELETE_WINDOW", lambda: (win.close(), root.destroy()))    root.mainloop()

In [ ]:
# Case 3 : Compilation de l'APK (TRES LONG : 30 min a 1 h, laissez tourner)
%cd /content/egm-os
!buildozer -v android debug

In [ ]:
# Case 4 : Telecharger l'APK sur votre ordinateur
import os
from google.colab import files
apk = '/content/egm-os/bin/' + os.listdir('/content/egm-os/bin')[0]
print('APK pret :', apk)
files.download(apk)